# Predicting F1 Pit Stops
**Author:** Koketso Raphasha | [Kaggle: Raphasha27](https://kaggle.com/Raphasha27)

**Playground Series - Season 6 Episode 5**

**Metric:** ROC AUC

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('/kaggle/input/playground-series-s6e5/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e5/test.csv')
print(f'Train: {train.shape} Test: {test.shape}')

## Feature Engineering

In [ ]:
y = train['PitStop']
cat_cols = train.select_dtypes(include='object').columns
for c in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[c], test[c]], axis=0).fillna('None').astype(str)
    le.fit(combined)
    train[c] = le.transform(train[c].fillna('None').astype(str))
    test[c] = le.transform(test[c].fillna('None').astype(str))
test_ids = test['id']
feat_cols = [c for c in train.columns if c not in ('id','PitStop','kfold')]
X = train[feat_cols].fillna(train[feat_cols].median())
Xt = test[feat_cols].fillna(train[feat_cols].median())
scaler = StandardScaler()
X_s = scaler.fit_transform(X)
Xt_s = scaler.transform(Xt)
print(f'Features: {X.shape[1]}')

## Model Training

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    'GB': GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=42),
    'RF': RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=4, random_state=42),
    'XGB': XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42),
}
for name, model in models.items():
    scores = cross_val_score(model, X_s, y, cv=cv, scoring='roc_auc')
    print(f'{name}: CV ROC AUC {scores.mean():.4f} (+/- {scores.std()*2:.4f})')
test_probs = np.zeros((Xt.shape[0], len(models)))
for i, (name, model) in enumerate(models.items()):
    model.fit(X_s, y)
    test_probs[:, i] = model.predict_proba(Xt_s)[:, 1]
preds = test_probs.mean(axis=1)
print(f'Predictions: min={preds.min():.4f}, max={preds.max():.4f}')

## Submission

In [ ]:
sub = pd.DataFrame({'id': test_ids, 'PitStop': preds})
sub.to_csv('submission.csv', index=False)
print('Ready: submission.csv')
sub.head()